# Quvurda noizotermik oqim — Python virtual laboratoriyasi
**Graetz masalasi · yashirin chekli ayirmalar · interaktiv Jupyter Notebook**

Bu notebook asl laboratoriyaning hisoblashlari va boshqaruvlarini Python, NumPy, Matplotlib va ipywidgets yordamida qayta yaratadi. Alohida HTTP server, IFrame, HTML sahifa yoki foydalanuvchi yozadigan JavaScript talab qilinmaydi. Standart Jupyter widgetlari brauzerda o‘zining odatiy frontend mexanizmiga ega; kichik HTML elementlari faqat sarlavha va qiymatlarni bezash uchun. Formulalar Matplotlib tasvirida chiziladi.

### Ishga tushirish
Terminalda notebook ishlaydigan Python muhitiga paketlarni o‘rnating:

```bash
python -m pip install jupyterlab ipywidgets numpy matplotlib pillow
python -m jupyterlab
```

JupyterLab ichida faylni ochib **Run → Run All Cells** ni tanlang. PyCharm'da ham notebookni shu interpreter bilan ishga tushirish mumkin; agar widgetlar ko‘rinmasa, JupyterLab brauzer interfeysidan foydalaning. Faqat odatiy Jupyter serveri ishlaydi; laboratoriya alohida server ochmaydi.

**Boshqaruv:** Re, Pr, kirish va devor harorati, L/D, radial N parametrlarini tanlang. Slayder qo‘yib yuborilganda qayta hisoblanadi. Kesimni siljiting yoki Play tugmasidan foydalaning. «Chekli ayirmalar / tugunlar» tabida j tugunni o‘zgartiring. CSV va GIF tugmalari fayllarni joriy ishchi katalogga saqlaydi.

## 1. Matematik model
Statsionar, laminar, siqilmaydigan suyuqlik; doimiy fizik xossalar, bo‘ylama diffuziya e’tiborsiz. Rivojlangan Puazeyl tezligi $w = u/U = 2(1-\eta^2)$, $v=0$.

$$s=x/D,\quad \eta=r/R,\quad Pe=Re\,Pr,\qquad wT_s=\frac4{Pe}(T_{\eta\eta}+\eta^{-1}T_\eta).$$

Kirishda $T_{0,j}=T_{in}$ ($j<N$), devorda $T_{i,N}=T_w$, markazda $T_\eta=0$. Burchakda devor sharti ustun. $h=1/N$, $\Delta s=(L/D)/M$, $A=4\Delta s/(Pe h^2)$.

Ichki tugunlarda, ixcham yozuvda:
$$a_jT_{i+1,j-1}+b_jT_{i+1,j}+c_jT_{i+1,j+1}=d_j.$$

To‘liq tenglama va markaz tenglamasi laboratoriyaning ikkinchi tabida bitta matematik tasvirda ko‘rsatiladi. Har bir tenglama bitta qatorda; tor oynada grafikni gorizontal siljiting.

Aralashma harorati: $T_b=\int_0^1 wT\eta\,d\eta/\int_0^1w\eta\,d\eta$. Thomas algoritmi har bir bo‘ylama kesimda uch diagonalli sistemani yechadi. Bu to‘liq bog‘langan Navye–Stoks masalasi emas. Animatsiya statsionar yechimning fazoviy kesimlarini ko‘rsatadi, fizik vaqtni emas.

## 2. Kutubxonalar va ma’lumotlar strukturasi

In [ ]:
from dataclasses import dataclass
from pathlib import Path
from io import BytesIO

import numpy as np
from matplotlib.figure import Figure
from matplotlib.backends.backend_agg import FigureCanvasAgg
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.ticker import MaxNLocator

BLUE = "#41629a"
RED = "#af5030"
INK = "#172c46"
MUTED = "#5a6b80"
GRID = "#dbe4ee"

TEMPERATURE_COLORS = LinearSegmentedColormap.from_list(
    "quvur",
    [BLUE, "#f1f4ee", "#a92f35"],
)


@dataclass(frozen=True)
class Parametrlar:
    reynolds: int = 500
    prandtl: float = 7
    inlet_temperature: float = 20
    wall_temperature: float = 80
    length_over_diameter: float = 250
    radial_steps: int = 40
    axial_steps: int = 300


@dataclass
class Natija:
    parameters: Parametrlar
    axial_coordinates: np.ndarray
    radial_coordinates: np.ndarray
    velocity_profile: np.ndarray
    temperature: np.ndarray
    bulk_temperature: np.ndarray
    diffusion_factor: float
    lower_diagonal: np.ndarray
    main_diagonal: np.ndarray
    upper_diagonal: np.ndarray

## 3. Chekli ayirmalar va Thomas algoritmi

In [ ]:
def validate_parameters(parameters):
    physical_values = [
        parameters.reynolds,
        parameters.prandtl,
        parameters.inlet_temperature,
        parameters.wall_temperature,
        parameters.length_over_diameter,
    ]
    if not np.all(np.isfinite(physical_values)):
        raise ValueError("Parametrlar chekli son bo‘lishi kerak.")

    positive_values = [
        parameters.reynolds,
        parameters.prandtl,
        parameters.length_over_diameter,
    ]
    if min(positive_values) <= 0:
        raise ValueError("Re, Pr va L/D musbat bo‘lishi kerak.")

    if parameters.radial_steps < 2 or parameters.axial_steps < 1:
        raise ValueError("N >= 2 va M >= 1 bo‘lishi kerak.")


def build_diagonals(velocity_profile, diffusion_factor):
    radial_steps = len(velocity_profile) - 1
    lower = np.zeros(radial_steps + 1)
    main = np.ones(radial_steps + 1)
    upper = np.zeros(radial_steps + 1)

    j = np.arange(1, radial_steps)
    A = diffusion_factor

    lower[j] = -A * (1 - 1 / (2 * j))
    main[j] = velocity_profile[j] + 2 * A
    upper[j] = -A * (1 + 1 / (2 * j))

    main[0] = velocity_profile[0] + 4 * A
    upper[0] = -4 * A

    # Devor: T_N = T_wall.
    lower[-1] = 0
    main[-1] = 1
    upper[-1] = 0
    return lower, main, upper


def factor_tridiagonal(lower, main, upper):
    modified_main = main.copy()
    multipliers = np.zeros_like(main)

    for j in range(1, len(main)):
        multipliers[j] = lower[j] / modified_main[j - 1]
        modified_main[j] -= multipliers[j] * upper[j - 1]

    return modified_main, multipliers


def solve_tridiagonal(right_hand_side, modified_main,
                      upper, multipliers):
    rhs = right_hand_side.copy()
    solution = np.empty_like(rhs)

    for j in range(1, len(rhs)):
        rhs[j] -= multipliers[j] * rhs[j - 1]

    solution[-1] = rhs[-1] / modified_main[-1]

    for j in range(len(rhs) - 2, -1, -1):
        solution[j] = (
            rhs[j] - upper[j] * solution[j + 1]
        ) / modified_main[j]

    return solution


def hisobla(parameters=Parametrlar()):
    """Graetz masalasini yashirin chekli ayirmalar bilan yechish."""
    validate_parameters(parameters)
    radial_steps = parameters.radial_steps
    axial_steps = parameters.axial_steps

    radial_coordinates = np.linspace(0, 1, radial_steps + 1)
    axial_coordinates = np.linspace(
        0, parameters.length_over_diameter, axial_steps + 1
    )
    radial_step = 1 / radial_steps
    axial_step = parameters.length_over_diameter / axial_steps
    peclet = parameters.reynolds * parameters.prandtl

    velocity_profile = 2 * (1 - radial_coordinates**2)
    diffusion_factor = 4 * axial_step / (peclet * radial_step**2)

    lower, main, upper = build_diagonals(
        velocity_profile, diffusion_factor
    )
    modified_main, multipliers = factor_tridiagonal(
        lower, main, upper
    )

    temperature = np.full(
        (axial_steps + 1, radial_steps + 1),
        float(parameters.inlet_temperature),
    )
    temperature[:, -1] = parameters.wall_temperature

    for i in range(1, axial_steps + 1):
        right_hand_side = velocity_profile * temperature[i - 1]
        right_hand_side[-1] = parameters.wall_temperature
        temperature[i] = solve_tridiagonal(
            right_hand_side, modified_main, upper, multipliers
        )

    weights = radial_coordinates * velocity_profile
    bulk_temperature = temperature @ weights / weights.sum()

    return Natija(
        parameters=parameters,
        axial_coordinates=axial_coordinates,
        radial_coordinates=radial_coordinates,
        velocity_profile=velocity_profile,
        temperature=temperature,
        bulk_temperature=bulk_temperature,
        diffusion_factor=diffusion_factor,
        lower_diagonal=lower,
        main_diagonal=main,
        upper_diagonal=upper,
    )

## 4. Qoldiq, tugun ma’lumotlari va CSV eksporti

In [ ]:
def qoldiq(result):
    temperature = result.temperature
    velocity = result.velocity_profile

    interior_residual = (
        result.lower_diagonal[1:-1] * temperature[1:, :-2]
        + result.main_diagonal[1:-1] * temperature[1:, 1:-1]
        + result.upper_diagonal[1:-1] * temperature[1:, 2:]
        - velocity[1:-1] * temperature[:-1, 1:-1]
    )
    center_residual = (
        result.main_diagonal[0] * temperature[1:, 0]
        + result.upper_diagonal[0] * temperature[1:, 1]
        - velocity[0] * temperature[:-1, 0]
    )
    wall_residual = (
        temperature[:, -1] - result.parameters.wall_temperature
    )
    return max(
        np.max(np.abs(interior_residual)),
        np.max(np.abs(center_residual)),
        np.max(np.abs(wall_residual)),
    )


def tugun(result, axial_index, radial_index):
    parameters = result.parameters
    i = int(np.clip(axial_index, 1, parameters.axial_steps))
    j = int(np.clip(radial_index, 1, parameters.radial_steps - 1))

    lower = result.lower_diagonal[j]
    main = result.main_diagonal[j]
    upper = result.upper_diagonal[j]
    previous = result.temperature[i - 1, j]
    right_hand_side = result.velocity_profile[j] * previous

    below, current, above = result.temperature[i, j - 1:j + 2]
    residual = (
        lower * below + main * current + upper * above
        - right_hand_side
    )
    return {
        "axial_index": i,
        "radial_index": j,
        "lower": lower,
        "main": main,
        "upper": upper,
        "right_hand_side": right_hand_side,
        "previous": previous,
        "below": below,
        "current": current,
        "above": above,
        "residual": residual,
    }


def csv_saqlash(result, filename="quvur_natijalar.csv"):
    radial_count = result.parameters.radial_steps + 1
    axial_count = result.parameters.axial_steps + 1

    table = np.column_stack([
        np.repeat(result.axial_coordinates, radial_count),
        np.tile(result.radial_coordinates, axial_count),
        result.temperature.ravel(),
        np.tile(result.velocity_profile, axial_count),
        np.repeat(result.bulk_temperature, radial_count),
    ])
    path = Path(filename).resolve()
    np.savetxt(
        path,
        table,
        delimiter=",",
        header="x_over_D,r_over_R,T_C,u_over_U,T_bulk_C",
        comments="",
        fmt="%.10g",
    )
    return path

## 5. Matematik nazorat

In [ ]:
test_cases = [
    Parametrlar(),
    Parametrlar(inlet_temperature=80, wall_temperature=20),
    Parametrlar(inlet_temperature=50, wall_temperature=50),
    Parametrlar(radial_steps=160),
]

for parameters in test_cases:
    test_result = hisobla(parameters)
    minimum = min(parameters.inlet_temperature,
                  parameters.wall_temperature)
    maximum = max(parameters.inlet_temperature,
                  parameters.wall_temperature)

    assert np.allclose(
        test_result.temperature[:, -1], parameters.wall_temperature
    )
    assert test_result.temperature.min() >= minimum - 1e-7
    assert test_result.temperature.max() <= maximum + 1e-7
    assert qoldiq(test_result) < 1e-7

result = hisobla()
assert np.isclose(result.temperature[90, 0], 22.2204660468)
assert np.isclose(result.bulk_temperature[90], 43.1463783837)

print(f"Markaz: {result.temperature[90, 0]:.2f} °C")
print(f"Aralashma: {result.bulk_temperature[90]:.2f} °C")
print(f"Maksimal diskret qoldiq: {qoldiq(result):.2e}")
print("Barcha matematik testlar o‘tdi.")

## 6. Grafiklar uchun yordamchi funksiyalar

In [ ]:
def create_figure(size):
    figure = Figure(
        figsize=size, facecolor="white", dpi=120,
        layout="constrained",
    )
    FigureCanvasAgg(figure)
    return figure


def style_axes(axes, show_grid=True):
    axes.set_facecolor("white")
    axes.tick_params(colors=MUTED, labelsize=9)
    axes.xaxis.label.set_color(MUTED)
    axes.yaxis.label.set_color(MUTED)
    for spine in axes.spines.values():
        spine.set_color(GRID)
    if show_grid:
        axes.grid(color=GRID, linewidth=0.8)
        axes.set_axisbelow(True)


def temperature_limits(result):
    lower, upper = sorted([
        result.parameters.inlet_temperature,
        result.parameters.wall_temperature,
    ])
    if lower == upper:
        return lower - 1, upper + 1
    return lower, upper


def style_title(axes, title):
    axes.set_title(title, color=INK, fontweight="bold")


def png(figure):
    buffer = BytesIO()
    figure.savefig(
        buffer, format="png", dpi=110, facecolor="white"
    )
    return buffer.getvalue()

## 7. Harorat maydoni, radial va bo‘ylama profillar

In [ ]:
def draw_temperature_field(figure, axes, result, i):
    style_axes(axes, show_grid=False)
    minimum, maximum = temperature_limits(result)
    distance = result.axial_coordinates[i]

    mirrored = np.concatenate((
        result.temperature[:, 1:][:, ::-1],
        result.temperature,
    ), axis=1)
    image = axes.imshow(
        mirrored.T,
        origin="lower",
        extent=(0, result.parameters.length_over_diameter, -1, 1),
        aspect="auto",
        cmap=TEMPERATURE_COLORS,
        norm=Normalize(minimum, maximum),
        interpolation="bilinear",
    )
    axes.axhline(0, color=MUTED, linestyle="--", linewidth=0.9)
    axes.axvline(distance, color=INK, linewidth=1.4)
    axes.set(
        yticks=[-1, 0, 1], yticklabels=["−R", "0", "+R"],
        xlabel="x/D",
    )
    style_title(axes, "Quvur ichidagi harorat maydoni")
    axes.annotate(
        f"{distance:.2f} D", xy=(distance, 1), xytext=(0, -5),
        textcoords="offset points", ha="center", va="top",
        color=INK, fontsize=9,
        bbox=dict(boxstyle="round,pad=.2", fc="white",
                  ec="none", alpha=0.85),
    )
    colorbar = figure.colorbar(
        image, ax=axes, orientation="horizontal",
        fraction=0.09, pad=0.12, aspect=40,
    )
    colorbar.set_label("Harorat, °C", color=MUTED, fontsize=9)
    colorbar.ax.tick_params(labelsize=8, colors=MUTED)
    colorbar.outline.set_visible(False)


def draw_radial_profiles(axes, result, i):
    style_axes(axes)
    minimum, maximum = temperature_limits(result)
    radius = result.radial_coordinates
    section = result.temperature[i]
    signed_radius = np.r_[-radius[:0:-1], radius]
    signed_temperature = np.r_[section[:0:-1], section]
    signed_velocity = 2 * (1 - signed_radius**2)

    axes.plot(signed_temperature, signed_radius,
              color=RED, linewidth=2.5)
    axes.axhline(0, color=GRID, linewidth=1)
    axes.set(
        xlim=(minimum, maximum), ylim=(-1, 1),
        ylabel="±r/R", xlabel="T, °C",
        yticks=[-1, -0.5, 0, 0.5, 1],
    )
    axes.xaxis.label.set_color(RED)
    axes.tick_params(axis="x", colors=RED)
    axes.xaxis.set_major_locator(MaxNLocator(5))
    style_title(axes, "Radial profillar")

    velocity_axes = axes.twiny()
    velocity_axes.plot(signed_velocity, signed_radius,
                       color=BLUE, linewidth=2.5)
    velocity_axes.set(
        xlim=(0, 2), xlabel="u/U", xticks=[0, 0.5, 1, 1.5, 2],
    )
    velocity_axes.xaxis.label.set_color(BLUE)
    velocity_axes.tick_params(colors=BLUE, labelsize=9)
    for spine in velocity_axes.spines.values():
        spine.set_color(GRID)


def draw_axial_profiles(axes, result, i):
    style_axes(axes)
    minimum, maximum = temperature_limits(result)
    distance = result.axial_coordinates[i]
    center_temperature = result.temperature[:, 0]
    axes.plot(result.axial_coordinates, result.bulk_temperature,
              color=RED, linewidth=2.5, label="Aralashma")
    axes.plot(result.axial_coordinates, center_temperature,
              color=BLUE, linewidth=2.5, label="Markaz")
    axes.axvline(distance, color=MUTED, linestyle="--", linewidth=1)
    axes.scatter(
        [distance, distance],
        [result.bulk_temperature[i], center_temperature[i]],
        color=[RED, BLUE], s=30, zorder=4,
    )
    axes.set(
        xlim=(0, result.parameters.length_over_diameter),
        ylim=(minimum, maximum), xlabel="x/D", ylabel="T, °C",
    )
    style_title(axes, "Bo‘ylama harorat")
    axes.legend(loc="best", frameon=False, fontsize=9)


def asosiy_grafik(result, axial_index=90):
    i = int(np.clip(axial_index, 0, result.parameters.axial_steps))
    figure = create_figure((11, 6.4))
    grid = figure.add_gridspec(2, 2, height_ratios=[1.0, 1.1])

    draw_temperature_field(figure, figure.add_subplot(grid[0, :]),
                           result, i)
    draw_radial_profiles(figure.add_subplot(grid[1, 0]), result, i)
    draw_axial_profiles(figure.add_subplot(grid[1, 1]), result, i)
    return figure

## 8. Chekli ayirma formulalari va tugunlar grafiklari

In [ ]:
def formula_grafik():
    # Har bir formula bitta MathText obyekti: avtomatik qator yo‘q.
    figure = Figure(figsize=(8.2, 2.65), facecolor="#edf3f8", dpi=120)
    FigureCanvasAgg(figure)
    formulas = [
        ("Chekli ayirma koeffitsiyenti",
         r"$A=\frac{4\Delta s}{Re\,Pr\,h^2}$"),
        ("Ichki tugun tenglamasi",
         r"$-A\left(1-\frac{1}{2j}\right)T_{i+1,j-1}"
         r"+(w_j+2A)T_{i+1,j}"
         r"-A\left(1+\frac{1}{2j}\right)T_{i+1,j+1}"
         r"=w_j T_{i,j}$"),
        ("Markazdagi tenglama",
         r"$(w_0+4A)T_{i+1,0}-4AT_{i+1,1}=w_0T_{i,0}$"),
    ]
    for row, (title, equation) in enumerate(formulas):
        top = 0.96 - row * 0.32
        figure.text(0.025, top, title, color=INK, fontsize=10,
                    fontweight="bold", va="top")
        figure.text(0.025, top - 0.12, equation, color=INK,
                    fontsize=16, va="top")
    return figure


def tugun_grafik(result, axial_index, radial_index):
    node = tugun(result, axial_index, radial_index)
    i = node["axial_index"]
    j = node["radial_index"]
    figure = create_figure((11, 3.0))
    axes = figure.add_subplot(111)
    next_label = f"i+2 = {i+1}"
    if i == result.parameters.axial_steps:
        next_label += " (sohadan tashqari)"

    axes.set(
        xlim=(-0.6, 2.65), ylim=(-1.5, 1.52),
        xticks=[0, 1, 2],
        xticklabels=[f"i = {i-1}", f"i+1 = {i}", next_label],
        yticks=[-1, 0, 1],
        yticklabels=[f"j−1 = {j-1}", f"j = {j}", f"j+1 = {j+1}"],
    )
    style_axes(axes)
    for spine in axes.spines.values():
        spine.set_visible(False)
    axes.tick_params(length=0)
    axes.plot([0, 1, 1], [0, 0, 1], color=BLUE, linewidth=3)
    axes.plot([1, 1], [0, -1], color=BLUE, linewidth=3)
    axes.scatter(
        [0, 1, 1], [0, 1, -1], s=180, facecolors="white",
        edgecolors=BLUE, linewidth=2.8, zorder=3,
    )
    axes.scatter([1], [0], s=190, color=RED, zorder=4)

    points = [
        (0, 0, "previous"), (1, 1, "above"),
        (1, 0, "current"), (1, -1, "below"),
    ]
    for x, y, key in points:
        vertical_offset = 10 if key == "previous" else 0
        axes.annotate(
            f"{node[key]:.4f} °C", (x, y),
            xytext=(13, vertical_offset), textcoords="offset points",
            ha="left", va="center", color=INK, fontsize=12,
        )
    style_title(axes, "Yashirin chekli ayirma shabloni")
    return figure


def tor_grafik(result, axial_index, radial_index):
    node = tugun(result, axial_index, radial_index)
    i = node["axial_index"]
    j = node["radial_index"]
    parameters = result.parameters
    figure = create_figure((10, 3.1))
    axes = figure.add_subplot(111)

    axial_indices = np.arange(
        max(0, i - 3), min(parameters.axial_steps, i + 3) + 1
    )
    radial_indices = np.arange(
        max(0, j - 3), min(parameters.radial_steps, j + 3) + 1
    )
    axial = result.axial_coordinates
    radial = result.radial_coordinates
    grid_x, grid_y = np.meshgrid(axial[axial_indices],
                                radial[radial_indices])
    values = result.temperature[np.ix_(axial_indices, radial_indices)].T
    minimum, maximum = temperature_limits(result)

    style_axes(axes)
    points = axes.scatter(
        grid_x, grid_y, c=values, cmap=TEMPERATURE_COLORS,
        vmin=minimum, vmax=maximum, s=42, edgecolors=GRID, zorder=3,
    )
    axes.scatter(axial[i], radial[j], s=140, facecolors="none",
                 edgecolors=INK, linewidth=2, zorder=4)
    axes.plot(axial[[i - 1, i]], radial[[j, j]],
              color=BLUE, linewidth=1.6)
    axes.plot(axial[[i, i]], radial[[j - 1, j + 1]],
              color=BLUE, linewidth=1.6)
    axes.set(
        xlabel="x/D", ylabel="r/R",
        xticks=axial[axial_indices], yticks=radial[radial_indices],
    )
    axes.tick_params(labelsize=8)
    style_title(axes, "Mahalliy hisoblash to‘ri • tugunlardagi harorat")
    figure.colorbar(points, ax=axes, label="°C", pad=0.02)
    return figure

## 9. GIF animatsiya

In [ ]:
def animatsiya_gif(result, filename="quvur_animatsiya_python.gif",
                   frames=31, duration=150):
    """Statsionar yechimning fazoviy kesimlari."""
    from PIL import Image

    frame_count = max(2, int(frames))
    indices = np.unique(np.linspace(
        0, result.parameters.axial_steps, frame_count, dtype=int
    ))
    images = []
    for i in indices:
        image_bytes = png(asosiy_grafik(result, i))
        with Image.open(BytesIO(image_bytes)) as image:
            images.append(image.convert("RGB"))

    path = Path(filename).resolve()
    images[0].save(
        path, save_all=True, append_images=images[1:],
        duration=duration, loop=0, optimize=True,
    )
    return path

## 10. Interaktiv laboratoriya

In [ ]:
import ipywidgets as widgets
from IPython.display import display, FileLink


def create_slider(widget_type, value, minimum, maximum, step,
                  description):
    return widget_type(
        value=value, min=minimum, max=maximum, step=step,
        description=description, continuous_update=False,
        style={"description_width": "initial"},
        layout=widgets.Layout(width="100%"),
    )


def create_image(maximum_width="1210px"):
    return widgets.Image(
        format="png",
        layout=widgets.Layout(width="100%", max_width=maximum_width),
    )


def create_button(description, icon, width):
    return widgets.Button(
        description=description, icon=icon,
        layout=widgets.Layout(width=width),
    )


class QuvurLaboratoriya:
    def __init__(self):
        self.updating = False
        self.result = hisobla()
        self.create_controls()
        self.create_outputs()
        self.build_layout()
        self.connect_events()
        self.render(all_tabs=True)

    def create_controls(self):
        self.reynolds_slider = create_slider(
            widgets.IntSlider, 500, 100, 2000, 50, "Reynolds, Re"
        )
        self.prandtl_slider = create_slider(
            widgets.FloatSlider, 7, 0.5, 15, 0.5, "Prandtl, Pr"
        )
        self.inlet_slider = create_slider(
            widgets.IntSlider, 20, 0, 100, 1, "Kirish T₀, °C"
        )
        self.wall_slider = create_slider(
            widgets.IntSlider, 80, 0, 100, 1, "Devor Tᵥ, °C"
        )
        self.length_slider = create_slider(
            widgets.IntSlider, 250, 20, 500, 10, "Uzunlik, L/D"
        )
        self.radial_dropdown = widgets.Dropdown(
            options=[10, 40, 80, 160], value=40,
            description="Radial N",
            style={"description_width": "initial"},
            layout=widgets.Layout(width="100%"),
        )
        self.parameter_controls = (
            self.reynolds_slider, self.prandtl_slider,
            self.inlet_slider, self.wall_slider,
            self.length_slider, self.radial_dropdown,
        )
        self.section_slider = create_slider(
            widgets.IntSlider, 90, 0, 300, 1, "Kesim indeksi i"
        )
        self.section_slider.readout = False
        self.node_slider = create_slider(
            widgets.IntSlider, 1, 1, 39, 1, "Ichki tugun j"
        )
        self.play = widgets.Play(
            value=90, min=0, max=300, step=5,
            interval=650, description="Animatsiya", repeat=False,
        )
        self.play_link = widgets.link(
            (self.play, "value"), (self.section_slider, "value")
        )
        self.reset_button = create_button(
            "Dastlabki qiymatlar", "refresh", "190px"
        )
        self.csv_button = create_button("CSV saqlash", "download", "145px")
        self.gif_button = create_button("GIF saqlash", "film", "145px")

    def create_outputs(self):
        self.metrics = widgets.HTML()
        self.model = widgets.HTML()
        self.summary = widgets.HTML()
        self.main_image = create_image()
        self.stencil_image = create_image()
        self.mesh_image = create_image("1100px")
        self.status = widgets.Output(
            layout=widgets.Layout(width="100%")
        )

        # O‘zgarmas kenglik: tenglama hech qachon qatorga bo‘linmaydi.
        self.formula_image = widgets.Image(
            value=png(formula_grafik()), format="png",
            layout=widgets.Layout(
                width="820px", min_width="820px",
                max_width="none", flex="0 0 auto",
            ),
        )
        self.formula_scroll = widgets.Box(
            [self.formula_image],
            layout=widgets.Layout(
                width="100%", max_width="100%", min_width="0px",
                overflow="auto hidden", display="flex",
            ),
        )
        self.formula_note = widgets.HTML(
            value="<small>Formula sig‘masa, gorizontal siljiting.</small>"
        )

    def build_layout(self):
        profiles_tab = widgets.VBox([self.main_image, self.model])
        nodes_tab = widgets.VBox(
            [self.formula_note, self.formula_scroll, self.node_slider,
             self.stencil_image, self.summary, self.mesh_image],
            layout=widgets.Layout(width="100%", min_width="0px"),
        )
        self.tabs = widgets.Tab(
            children=[profiles_tab, nodes_tab],
            layout=widgets.Layout(width="100%", min_width="0px"),
        )
        self.tabs.set_title(0, "Harorat va tezlik")
        self.tabs.set_title(1, "Chekli ayirmalar / tugunlar")

        controls = widgets.GridBox(
            list(self.parameter_controls),
            layout=widgets.Layout(
                grid_template_columns="repeat(auto-fit, minmax(270px, 1fr))",
                grid_gap="8px 20px", width="100%", padding="12px",
                border="1px solid #dbe4ee",
            ),
        )
        buttons = widgets.HBox(
            [self.play, self.reset_button, self.csv_button, self.gif_button],
            layout=widgets.Layout(
                flex_flow="row wrap", grid_gap="8px", align_items="center"
            ),
        )
        heading = widgets.HTML(
            value=(
                '<h2 style="color:#41629a;margin:4px 0">'
                'Quvurda noizotermik oqim</h2>'
                '<p>Python · Graetz masalasi · yashirin chekli ayirmalar</p>'
            )
        )
        self.view = widgets.VBox(
            [heading, controls, self.metrics, self.section_slider,
             buttons, self.tabs, self.status],
            layout=widgets.Layout(
                width="100%", max_width="1250px", min_width="0px",
                padding="8px",
            ),
        )

    def connect_events(self):
        for control in self.parameter_controls:
            control.observe(self.on_parameters_changed, names="value")
        self.section_slider.observe(self.on_section_changed, names="value")
        self.node_slider.observe(self.on_node_changed, names="value")
        self.tabs.observe(self.on_tab_changed, names="selected_index")
        self.reset_button.on_click(self.reset)
        self.csv_button.on_click(self.export_csv)
        self.gif_button.on_click(self.export_gif)

    def read_parameters(self):
        return Parametrlar(
            reynolds=self.reynolds_slider.value,
            prandtl=self.prandtl_slider.value,
            inlet_temperature=self.inlet_slider.value,
            wall_temperature=self.wall_slider.value,
            length_over_diameter=self.length_slider.value,
            radial_steps=self.radial_dropdown.value,
        )

    def on_parameters_changed(self, change=None):
        if self.updating:
            return
        self.updating = True
        try:
            self.result = hisobla(self.read_parameters())
            self.node_slider.max = self.result.parameters.radial_steps - 1
            self.node_slider.value = min(
                self.node_slider.value, self.node_slider.max
            )
        finally:
            self.updating = False
        self.render()

    def on_section_changed(self, change=None):
        if not self.updating:
            self.render()

    def on_node_changed(self, change=None):
        if not self.updating:
            self.render_nodes()

    def on_tab_changed(self, change=None):
        if not self.updating:
            self.render()

    def render_metrics(self, i):
        result = self.result
        parameters = result.parameters
        peclet = parameters.reynolds * parameters.prandtl
        items = [
            ("Peclet, Re·Pr", f"{peclet:g}"),
            ("Markaz T, °C", f"{result.temperature[i, 0]:.2f}"),
            ("Aralashma T, °C", f"{result.bulk_temperature[i]:.2f}"),
            ("Kesim, x/D", f"{result.axial_coordinates[i]:.2f}"),
        ]
        boxes = []
        for label, value in items:
            boxes.append(
                '<div style="background:#edf3f8;border-radius:9px;'
                'padding:12px;min-width:120px;flex:1;color:#53677e">'
                f'{label}<div style="font-size:25px;font-weight:700;'
                f'color:#172c46">{value}</div></div>'
            )
        cards = "".join(boxes)
        self.metrics.value = (
            '<div style="display:flex;gap:8px;flex-wrap:wrap;'
            f'margin:10px 0">{cards}</div>'
        )
        self.section_slider.description = (
            f"Kesim: {result.axial_coordinates[i]:.2f} D (i={i})"
        )
        axial_step = parameters.length_over_diameter / parameters.axial_steps
        radial_step = 1 / parameters.radial_steps
        warning = ""
        if peclet < 100:
            warning = " | Pe kichik: bo‘ylama diffuziya muhim bo‘lishi mumkin."
        self.model.value = (
            f"<p>Model: laminar, doimiy xossalar; Δs = {axial_step:.4f}, "
            f"h = {radial_step:.4f}; A = {result.diffusion_factor:.6f}"
            f"{warning}</p>"
            "<p>Ko‘k: tezlik/markaz; qizil: harorat/aralashma. "
            "Animatsiya fazoviy kesimlarni ko‘rsatadi.</p>"
        )

    def render(self, all_tabs=False):
        i = self.section_slider.value
        self.render_metrics(i)
        if all_tabs or self.tabs.selected_index == 0:
            self.main_image.value = png(asosiy_grafik(self.result, i))
        if all_tabs or self.tabs.selected_index == 1:
            self.render_nodes()

    def render_nodes(self):
        result = self.result
        node = tugun(result, self.section_slider.value,
                     self.node_slider.value)
        i = node["axial_index"]
        j = node["radial_index"]
        self.stencil_image.value = png(tugun_grafik(result, i, j))
        self.mesh_image.value = png(tor_grafik(result, i, j))

        equation = (
            f'({node["lower"]:.4f})T<sub>{j-1}</sub> + '
            f'({node["main"]:.4f})T<sub>{j}</sub> + '
            f'({node["upper"]:.4f})T<sub>{j+1}</sub> = '
            f'{node["right_hand_side"]:.4f}'
        )
        self.summary.value = (
            '<div style="background:#edf3f8;color:#172c46;padding:12px;'
            'border-left:4px solid #41629a">'
            '<div style="overflow-x:auto;white-space:nowrap">'
            f'{equation}</div><b>Hisoblangan T<sub>{j}</sub> = '
            f'{node["current"]:.4f} °C</b> · '
            f'Qoldiq: {node["residual"]:.2e}<br>'
            f'A = {result.diffusion_factor:.6f}; '
            f'kesim: {result.axial_coordinates[i]:.4f} D</div>'
            '<p>i = 0 tanlansa birinchi hisoblash qadami ko‘rsatiladi. '
            'Qoldiq yaxlitlanmagan koeffitsiyentlardan hisoblanadi.</p>'
        )

    def reset(self, button=None):
        self.updating = True
        try:
            if self.play.has_trait("_playing"):
                self.play._playing = False
            defaults = (500, 7, 20, 80, 250, 40)
            for control, value in zip(self.parameter_controls, defaults):
                control.value = value
            self.section_slider.value = 90
            self.node_slider.max = 39
            self.node_slider.value = 1
        finally:
            self.updating = False
        self.on_parameters_changed()

    def export_csv(self, button=None):
        with self.status:
            self.status.clear_output(wait=True)
            path = csv_saqlash(self.result)
            print(f"Saqlandi: {path}")
            display(FileLink(path.name))

    def export_gif(self, button=None):
        self.gif_button.disabled = True
        with self.status:
            self.status.clear_output(wait=True)
            try:
                print("GIF kadrlarini hisoblash va saqlash...")
                path = animatsiya_gif(self.result)
                print(f"Saqlandi: {path}")
                display(FileLink(path.name))
            finally:
                self.gif_button.disabled = False

    def close(self):
        self.play_link.unlink()
        # Faqat laboratoriya vidjetlarini rekursiv yopamiz.
        visited = set()

        def close_widget(widget):
            if id(widget) in visited:
                return
            visited.add(id(widget))
            for child in getattr(widget, "children", ()):
                close_widget(child)
            widget.close()

        close_widget(self.view)


if "lab" in globals():
    lab.close()
lab = QuvurLaboratoriya()
display(lab.view)

## 11. Mustaqil tajribalar (ixtiyoriy)
Quyidagi misollarni yangi katakka ko‘chiring.

```python
parameters = Parametrlar(reynolds=1000, radial_steps=80)
result = hisobla(parameters)
print(result.temperature.shape, qoldiq(result))

from IPython.display import Image, display
display(Image(data=png(asosiy_grafik(result, 90))))
display(Image(data=png(tugun_grafik(result, 90, 5))))
display(Image(data=png(tor_grafik(result, 90, 5))))
```

```python
path = animatsiya_gif(result)
display(Image(filename=str(path)))
print(csv_saqlash(result))
```

Ayni widgetlar natijasi: `lab.result.temperature`, `lab.result.bulk_temperature`, `lab.result.axial_coordinates`, `lab.result.radial_coordinates`, `lab.result.velocity_profile`.

**Aniqlik:** to‘rni zichlashtirib natijani taqqoslang. Kichik Pe da bo‘ylama diffuziyani tashlab ketish yetarli asoslanmasligi mumkin. Qoldiq kichikligi algebraik yechimning mosligini ko‘rsatadi, diskretlashtirish xatosining nol ekanini emas.